Notebook to evaluate the impact of different PC Inference Settings

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import os
import gc
import math
import random
from typing import Union
from tqdm import tqdm
import wandb
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler
from transformers import AutoTokenizer
from accelerate.utils import ProjectConfiguration, set_seed
from accelerate import Accelerator
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    ControlNetModel,
    UNet2DConditionModel,
)
from torch.utils.data import DataLoader
import torch
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

In [ ]:
from src.t2iadapter.config import (T2IConfig)
from src.t2iadapter.MRIProjector import MRIProjector, LatentMRIProjector
from src.t2iadapter.utils import (
    import_model_class_from_model_name_or_path,
    compute_embeddings_sd1x5,
    print_trainable_parameters,
    generate_mri_slices_partial_latent_align_dc_lora,
    generate_mri_slices_partial_latent_align_dc,
    generate_mri_slices_partial_latent_align_dc_no_t2i
)
from src.slicedMRI import DatasetConfig, FastMRILazyDataset
from src.slicedMRI.transform_to_2D_slices import build_fastMRI_manifest
from src.eval import MRIEvaluator
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict

# LoRA PC Eval

In [ ]:
train_val_test_split = (0.8, 0.1, 0.1)
assert sum(train_val_test_split) == 1.0, "Dataset split should sum up to one"

shared_config: dict = {
    "data_dir": Path("./fastMRI_brain_DICOM"),
    "mode": "train",
    "fractions": train_val_test_split,
    "target_size": (512, 512),
    "contrast_filter": "T2",
    "strength_filter": "3.0T",
    "scale_factor": 4.0,
    "fastMRI_manifest_json": "./fast_MRI_brain_patient_records_manifest_small.json",
}
config = DatasetConfig(**shared_config)
t2i_config = T2IConfig(
    train_batch_size=32,
    test_batch_size=16,
    partial_start_step=700, # sd 1.5 has a noise range of [0,1000]
    max_train_steps=6500,
    pretrained_vae_model_name_or_path="microsoft/mri-autoencoder-v0.1",
)
shared_config["mode"] = "test"
test_dataset = FastMRILazyDataset(config=DatasetConfig(**shared_config))
test_loader = DataLoader(
    test_dataset,
    batch_size=t2i_config.test_batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
accelerator_project_config = ProjectConfiguration(
    project_dir=t2i_config.output_dir, logging_dir=t2i_config.logging_dir
)
accelerator = Accelerator(
    gradient_accumulation_steps=t2i_config.gradient_accumulation_steps,
    mixed_precision=t2i_config.mixed_precision,
    log_with=t2i_config.report_to,
    project_config=accelerator_project_config,
)
set_seed(t2i_config.seed)
os.makedirs(t2i_config.output_dir, exist_ok=True)

weight_dtype = torch.float32
if accelerator.mixed_precision == "fp16":
    weight_dtype = torch.float16
elif accelerator.mixed_precision == "bf16":
    weight_dtype = torch.bfloat16
tokenizer = AutoTokenizer.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="tokenizer",
    revision=t2i_config.revision,
    use_fast=False,
)
text_encoder_cls = import_model_class_from_model_name_or_path(
    t2i_config.pretrained_model_name_or_path,
    t2i_config.revision,
    subfolder="text_encoder",
)
noise_scheduler = DDPMScheduler.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="scheduler",
    prediction_type=t2i_config.ddpm_scheduler_prediction_type,
    timestep_spacing=t2i_config.ddpm_scheduler_timestep_spacing,
    rescale_betas_zero_snr=t2i_config.ddpm_scheduler_rescale_betas_zero_snr,
)
text_encoder = text_encoder_cls.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="text_encoder",
    revision=t2i_config.revision,
    torch_dtype=weight_dtype,
)
vae_encoder = AutoencoderKL.from_pretrained(
    t2i_config.pretrained_vae_model_name_or_path,
    subfolder=None,
    revision=t2i_config.revision,
    torch_dtype=weight_dtype,
)
vae_decoder = AutoencoderKL.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="vae",
    revision=t2i_config.revision,
    torch_dtype=weight_dtype,
)
unet = UNet2DConditionModel.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="unet",
    revision=t2i_config.revision,
    torch_dtype=weight_dtype,
)
vae_encoder.requires_grad_(False)
vae_decoder.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

print(
    f"Using VAE-Encoder: {vae_encoder.config['_name_or_path']}, VAE-Decoder: {vae_decoder.config['_name_or_path']}, UNET: {unet.config['_name_or_path']}, Scheduler Steps: {noise_scheduler.config['num_train_timesteps']}"
)
if t2i_config.enable_xformers_memory_efficient_attention:
    import xformers  # pyright: ignore[reportMissingImports]
    from packaging import version
    xformers_version = version.parse(xformers.__version__)
    if xformers_version == version.parse("0.0.16"):
        print(
            "xFormers 0.0.16 cannot be used for training in some GPUs. If you observe problems during training, please update xFormers to at least 0.0.17. See https://huggingface.co/docs/diffusers/main/en/optimization/xformers for more details."
        )
    unet.enable_xformers_memory_efficient_attention()
    print(f"Enabled efficient attention on UNET")
if t2i_config.gradient_checkpointing:
    unet.enable_gradient_checkpointing()
    print(f"Enabled gradient checkpointing on UNET")
if t2i_config.allow_tf32:
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"Enabled matmul.allow_tf32")
if t2i_config.scale_lr:
    learning_rate = (
        t2i_config.learning_rate
        * t2i_config.gradient_accumulation_steps
        * t2i_config.train_batch_size
        * accelerator.num_processes
    )
else:
    learning_rate = t2i_config.learning_rate
if t2i_config.use_8bit_adam:
    try:
        import bitsandbytes as bnb  # pyright: ignore[reportMissingImports]
    except ImportError:
        raise ImportError(
            "To use 8-bit Adam, please install the bitsandbytes library: `pip install bitsandbytes`."
        )
    optimizer_class = bnb.optim.AdamW8bit
    print(f"Enabled 8bit adam")
else:
    optimizer_class = torch.optim.AdamW
is_special_vae = (
    t2i_config.pretrained_vae_model_name_or_path == "microsoft/mri-autoencoder-v0.1"
)
output_dims = 2 if is_special_vae else 3

# LoRA config — inject low-rank matrices into UNet attention layers
unet_lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    init_lora_weights="gaussian",
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
)
unet.add_adapter(unet_lora_config)

latent_projector = LatentMRIProjector(
    spatial_in=128, spatial_out=64, in_channels=4, out_channels=4
).to(accelerator.device)
mri_projector = MRIProjector(output_dims=output_dims).to(accelerator.device)

# Optimizer targets: LoRA params (requires_grad=True on UNet) + projectors
params_to_optimize = (
    list(filter(lambda p: p.requires_grad, unet.parameters()))
    + list(latent_projector.parameters())
    + list(mri_projector.parameters())
)
optimizer = optimizer_class(
    params_to_optimize,
    lr=learning_rate,
    betas=(t2i_config.adam_beta1, t2i_config.adam_beta2),
    weight_decay=t2i_config.adam_weight_decay,
    eps=t2i_config.adam_epsilon,
)

# For mixed precision training we cast the text_encoder and vae weights to half-precision
# as these models are only used for inference, keeping weights in full precision is not required.
weight_dtype = torch.float32
if accelerator.mixed_precision == "fp16":
    weight_dtype = torch.float16
elif accelerator.mixed_precision == "bf16":
    weight_dtype = torch.bfloat16
# Move vae, unet and text_encoder to device and cast to weight_dtype
vae_encoder.to(accelerator.device, dtype=weight_dtype)
vae_decoder.to(accelerator.device, dtype=weight_dtype)
unet.to(accelerator.device, dtype=weight_dtype)
print("Models loaded. UNet and VAE in", weight_dtype)

print_trainable_parameters(unet, name="UNet (LoRA)")
print_trainable_parameters(latent_projector, name="MRI Latent Projector")
print_trainable_parameters(mri_projector, name="MRI-Projector")
# Scheduler and math around the number of training steps.
tokenizers = [tokenizer]
gc.collect()
torch.cuda.empty_cache()
num_update_steps_per_epoch = math.ceil(1e7 / t2i_config.gradient_accumulation_steps)
if t2i_config.max_train_steps is None:
    t2i_config.max_train_steps = (
        t2i_config.num_train_epochs * num_update_steps_per_epoch
    )
lr_scheduler = get_scheduler(
    t2i_config.lr_scheduler_name,
    optimizer=optimizer,
    num_warmup_steps=t2i_config.lr_warmup_steps * accelerator.num_processes,
    num_training_steps=t2i_config.max_train_steps * accelerator.num_processes,
    num_cycles=t2i_config.lr_num_cycles,
    power=t2i_config.lr_power,
)
ema_unet = EMAModel(
    filter(lambda p: p.requires_grad, unet.parameters()),
    decay=0.9999,
)
# Prepare everything with our `accelerator`.
# UNet is included because it has trainable LoRA params
unet, latent_projector, mri_projector, optimizer, lr_scheduler = accelerator.prepare(
    unet, latent_projector, mri_projector, optimizer, lr_scheduler
)

In [ ]:
from peft import set_peft_model_state_dict

output_dir = "/content/drive/MyDrive/Colab Notebooks/MasterInfo/GenAI/lora_partial_65k"
mri_projector.load_state_dict(torch.load(os.path.join(output_dir, "mri_projector.pt")))
latent_projector.load_state_dict(torch.load(os.path.join(output_dir, "latent_projector.pt")))
lora_state_dict = torch.load(os.path.join(output_dir, "unet_lora_tuned.pt"))
set_peft_model_state_dict(unet, lora_state_dict)
gc.collect()
torch.cuda.empty_cache()
print("Loaded")

Loaded


In [ ]:
from itertools import product

psnr = PeakSignalNoiseRatio(data_range=1.0).to(accelerator.device)
ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(accelerator.device)

dcs = [1.5, 3.0, 4.5]
tapers = [0.15, 0.3, 0.45]
configs = product(dcs, tapers, [True])
configs = [x for x in configs]
configs.append([1.5, 0.45, False])
for taper, reduction_factor, use_data_consistency in configs:
  print(f"LoRA: evaluating now tau: {taper}, r: {reduction_factor}, use: {use_data_consistency}")
  metrics = []
  for idx, batch in tqdm(enumerate(test_loader), disable=True):
      with torch.no_grad():
          prompt_embeds_eval = compute_embeddings_sd1x5(
              batch=batch,
              proportion_empty_prompts=0.0,  # Use 0.0 for evaluation when skipping CFG
              text_encoders=[text_encoder],
              tokenizers=tokenizers,
              accelerator=accelerator,
          )["prompt_embeds"]
      images_batch_np, postprocessed = generate_mri_slices_partial_latent_align_dc_lora(
          batch=batch,
          mri_projector=mri_projector,
          latent_projector=latent_projector,
          unet=unet,
          vae_encoder=vae_encoder,
          vae_decoder=vae_decoder,
          noise_scheduler=noise_scheduler,
          prompt_embeds=prompt_embeds_eval,
          start_step=t2i_config.partial_start_step,
          num_inference_steps=500,
          weight_dtype=weight_dtype,
          accelerator=accelerator,
          use_data_consistency=use_data_consistency,
          dc_reduction_factor=reduction_factor,
          taper=taper,
          apply_final_pixel_dc=True,
      )
      images_batch_np = 1 - images_batch_np
      channels = images_batch_np.shape[3]
      if batch["hr"].ndim == 3:
          gt = (
              batch["hr"]
              .unsqueeze(1)
              .expand((t2i_config.test_batch_size, channels, 512, 512))
              .to(accelerator.device)
          )
      else:
          gt = (
              batch["hr"]
              .expand((t2i_config.test_batch_size, channels, 512, 512))
              .to(accelerator.device)
          )
      metrics_batch = MRIEvaluator.eval_all_metrics(
          ground_truth=gt,
          generated=torch.from_numpy(images_batch_np)
          .permute(0, 3, 1, 2)
          .to(accelerator.device),
          psnr=psnr,
          ssim=ssim,
      )  # returns hfen, nmse, psnr, ssim
      metrics.append(metrics_batch)
      if idx == 9:
        break

metrics_np = np.array(metrics)  # shape: (num_batches, 4)
avg_metrics = metrics_np.mean(axis=0)
print(f"Metrics are {avg_metrics}")
# hfen, nmse, psnr, ssim

# T2I Adapter PC Eval

In [ ]:
from src.t2iadapter.t2iadapter import (
    Adapter_XL,
)

train_val_test_split = (0.8, 0.1, 0.1)
assert sum(train_val_test_split) == 1.0, "Dataset split should sum up to one"

shared_config: dict = {
    "data_dir": Path("./fastMRI_brain_DICOM"),
    "mode": "train",
    "fractions": train_val_test_split,
    "target_size": (512, 512),
    "contrast_filter": "T2",
    "strength_filter": "3.0T",
    "scale_factor": 4.0,
    "fastMRI_manifest_json": "./fast_MRI_brain_patient_records_manifest_small.json",
}
config = DatasetConfig(**shared_config)
t2i_config = T2IConfig(
    train_batch_size=32,
    test_batch_size=16,
    partial_start_step=700, # sd 1.5 has a noise range of [0,1000]
    max_train_steps=6500,
    pretrained_vae_model_name_or_path="microsoft/mri-autoencoder-v0.1",
)
shared_config["mode"] = "test"
test_dataset = FastMRILazyDataset(config=DatasetConfig(**shared_config))
test_loader = DataLoader(
    test_dataset,
    batch_size=t2i_config.test_batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
logging_dir = Path(t2i_config.output_dir, t2i_config.logging_dir)
accelerator_project_config = ProjectConfiguration(
    project_dir=t2i_config.output_dir, logging_dir=t2i_config.logging_dir
)
accelerator = Accelerator(
    gradient_accumulation_steps=t2i_config.gradient_accumulation_steps,
    mixed_precision=t2i_config.mixed_precision, # set bf16
    log_with=t2i_config.report_to,
    project_config=accelerator_project_config,
)
set_seed(t2i_config.seed)
os.makedirs(t2i_config.output_dir, exist_ok=True)

weight_dtype = torch.float32
if accelerator.mixed_precision == "fp16":
    weight_dtype = torch.float16
elif accelerator.mixed_precision == "bf16":
    weight_dtype = torch.bfloat16
# load the tokenizers
tokenizer = AutoTokenizer.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="tokenizer",
    revision=t2i_config.revision,
    use_fast=False,
)
# load the correct scheduler and models
text_encoder_cls = import_model_class_from_model_name_or_path(
    t2i_config.pretrained_model_name_or_path,
    t2i_config.revision,
    subfolder="text_encoder",
)
# Load scheduler and models
# timesteps defauls to 1000
noise_scheduler = DDPMScheduler.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="scheduler",
    prediction_type=t2i_config.ddpm_scheduler_prediction_type,  # velocity prediction
    timestep_spacing=t2i_config.ddpm_scheduler_timestep_spacing,  # for zero-SNR
    rescale_betas_zero_snr=t2i_config.ddpm_scheduler_rescale_betas_zero_snr,  # enforces pure noise at t=1000
)
text_encoder = text_encoder_cls.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="text_encoder",
    revision=t2i_config.revision,
    torch_dtype=weight_dtype
)
vae_path = (
    t2i_config.pretrained_model_name_or_path
    if t2i_config.pretrained_vae_model_name_or_path is None
    else t2i_config.pretrained_vae_model_name_or_path
)
vae_encoder = AutoencoderKL.from_pretrained(
    t2i_config.pretrained_vae_model_name_or_path,
    subfolder=None,
    revision=t2i_config.revision,
    torch_dtype=weight_dtype,
)
vae_decoder = AutoencoderKL.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="vae",
    revision=t2i_config.revision,
    torch_dtype=weight_dtype,
)
unet = UNet2DConditionModel.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="unet",
    revision=t2i_config.revision,
    torch_dtype=weight_dtype,
)
# These are never trained to circumvent mode collapse (see controlnet paper)
vae_encoder.requires_grad_(False)
vae_decoder.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

print(
    f"Using VAE-Encoder: {vae_encoder.config['_name_or_path']}, VAE-Decoder: {vae_decoder.config['_name_or_path']}, UNET: {unet.config['_name_or_path']}, Scheduler Steps: {noise_scheduler.config["num_train_timesteps"]}"
)
if t2i_config.enable_xformers_memory_efficient_attention:
    import xformers # pyright: ignore[reportMissingImports]
    from packaging import version
    xformers_version = version.parse(xformers.__version__)
    if xformers_version == version.parse("0.0.16"):
        print(
            "xFormers 0.0.16 cannot be used for training in some GPUs. If you observe problems during training, please update xFormers to at least 0.0.17. See https://huggingface.co/docs/diffusers/main/en/optimization/xformers for more details."
        )
    unet.enable_xformers_memory_efficient_attention()
    print(f"Enabled efficient attention on UNET")
if t2i_config.gradient_checkpointing:
    unet.enable_gradient_checkpointing()
    print(f"Enabled gradient checkpointing on UNET")
# Enable TF32 for faster training on Ampere GPUs,
# cf https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
if t2i_config.allow_tf32:
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"Enabled matmul.allow_tf32")
if t2i_config.scale_lr:
    learning_rate = (
        t2i_config.learning_rate
        * t2i_config.gradient_accumulation_steps
        * t2i_config.train_batch_size
        * accelerator.num_processes
    )
else:
    learning_rate = t2i_config.learning_rate
# Use 8-bit Adam for lower memory usage or to fine-tune the model in 16GB GPUs
if t2i_config.use_8bit_adam:
    try:
        import bitsandbytes as bnb # pyright: ignore[reportMissingImports]
    except ImportError:
        raise ImportError(
            "To use 8-bit Adam, please install the bitsandbytes library: `pip install bitsandbytes`."
        )
    optimizer_class = bnb.optim.AdamW8bit
    print(f"Enabled 8bit adam")
else:
    optimizer_class = torch.optim.AdamW
class CheckpointedModel(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return checkpoint(self.model, x)


is_special_vae = (
    t2i_config.pretrained_vae_model_name_or_path == "microsoft/mri-autoencoder-v0.1"
)
output_dims = 2 if is_special_vae else 3
adapter = CheckpointedModel(
    Adapter_XL(
        channels=[320, 640, 1280, 1280],
        nums_rb=3,
        cin=output_dims * 64,
        ksize=3,
        sk=True,
        use_conv=True,
    )
).to(accelerator.device)
latent_projector = LatentMRIProjector(
    spatial_in=128, spatial_out=64, in_channels=4, out_channels=4
).to(accelerator.device)
mri_projector = MRIProjector(output_dims=output_dims).to(accelerator.device)
params_to_optimize = (
    list(adapter.parameters())
    + list(latent_projector.parameters())
    + list(mri_projector.parameters())
)
optimizer = optimizer_class(
    params_to_optimize,
    lr=learning_rate,
    betas=(t2i_config.adam_beta1, t2i_config.adam_beta2),
    weight_decay=t2i_config.adam_weight_decay,
    eps=t2i_config.adam_epsilon,
)

# For mixed precision training we cast the text_encoder and vae weights to half-precision
# as these models are only used for inference, keeping weights in full precision is not required.
weight_dtype = torch.float32
if accelerator.mixed_precision == "fp16":
    weight_dtype = torch.float16
elif accelerator.mixed_precision == "bf16":
    weight_dtype = torch.bfloat16
# Move vae, unet and text_encoder to device and cast to weight_dtype
# The VAE is in float32 to avoid NaN losses.
#if t2i_config.pretrained_vae_model_name_or_path is not None:
vae_encoder.to(accelerator.device, dtype=weight_dtype)
vae_decoder.to(accelerator.device, dtype=weight_dtype)
#else:
#    vae_encoder.to(accelerator.device, dtype=torch.float32)
unet.to(accelerator.device, dtype=weight_dtype)
# text_encoder.to(accelerator.device, dtype=weight_dtype)
print("Models loaded. UNet and VAE in", weight_dtype)
# Let's first compute all the embeddings so that we can free up the text encoders from memory.
# text_encoders = [text_encoder]
tokenizers = [tokenizer]
gc.collect()
torch.cuda.empty_cache()
# Scheduler and math around the number of training steps.
# num_update_steps_per_epoch = math.ceil(len(train_dataloader) / args.gradient_accumulation_steps)
num_update_steps_per_epoch = math.ceil(1e7 / t2i_config.gradient_accumulation_steps)
if t2i_config.max_train_steps is None:
    t2i_config.max_train_steps = (
        t2i_config.num_train_epochs * num_update_steps_per_epoch
    )
lr_scheduler = get_scheduler(
    t2i_config.lr_scheduler_name,
    optimizer=optimizer,
    num_warmup_steps=t2i_config.lr_warmup_steps * accelerator.num_processes,
    num_training_steps=t2i_config.max_train_steps * accelerator.num_processes,
    num_cycles=t2i_config.lr_num_cycles,
    power=t2i_config.lr_power,
)
ema_adapter = EMAModel(
    adapter.parameters(),
    model_cls=adapter.__class__,  # Custom class, passing it here helps with some utilities
    decay=0.9999,
)
# Prepare everything with our `accelerator`.
adapter, latent_projector, mri_projector, unet, optimizer, lr_scheduler = accelerator.prepare(
    adapter, latent_projector, mri_projector, unet, optimizer, lr_scheduler
)

print_trainable_parameters(latent_projector, name="MRI Latent Projector")
print_trainable_parameters(adapter, name="T2I-Adapter")
print_trainable_parameters(mri_projector, name="MRI-Projector")

In [ ]:
output_dir = "/content/drive/MyDrive/Colab Notebooks/MasterInfo/GenAI/T2I + Finetune + Partial Diffusion + B48 + 9,9k steps (cosmic-dragon-83)"
mri_projector.load_state_dict(torch.load(os.path.join(output_dir, "mri_projector_sun.pt")))
latent_projector.load_state_dict(torch.load(os.path.join(output_dir, "inverse_latent_projector_sun.pt")))
adapter.load_state_dict(torch.load(os.path.join(output_dir, "adapter_tuned_sun.pt")))
gc.collect()
torch.cuda.empty_cache()

In [ ]:
psnr = PeakSignalNoiseRatio(data_range=1.0).to(accelerator.device)
ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(accelerator.device)

from itertools import product

dcs = [1.5, 3.0, 4.5]
tapers = [0.15, 0.3, 0.45]
configs = product(dcs, tapers, [True])
configs = [x for x in configs]
configs.append([1.5, 0.45, False])
for taper, reduction_factor, use_data_consistency in configs:
  print(f"T2I: evaluating now tau: {taper}, r: {reduction_factor}, use: {use_data_consistency}")
  metrics = []
  for idx, batch in tqdm(enumerate(test_loader), disable=True):
      with torch.no_grad():
          prompt_embeds_eval = compute_embeddings_sd1x5(
              batch=batch,
              proportion_empty_prompts=0.0,  # Use 0.0 for evaluation when skipping CFG
              text_encoders=[text_encoder],
              tokenizers=tokenizers,
              accelerator=accelerator,
          )["prompt_embeds"]
      image_batch_np, postprocessed = generate_mri_slices_partial_latent_align_dc(
          batch=batch,
          adapter=adapter,
          mri_projector=mri_projector,
          latent_projector=latent_projector,
          unet=unet,
          vae_encoder=vae_encoder,
          vae_decoder=vae_decoder,
          noise_scheduler=noise_scheduler,
          prompt_embeds=prompt_embeds_eval,
          start_step=t2i_config.partial_start_step,  # Start from t=250
          num_inference_steps=500,  # Scheduler will slice this
          weight_dtype=weight_dtype,
          accelerator=accelerator,
          use_data_consistency=use_data_consistency,
          dc_reduction_factor=reduction_factor,
          taper=taper,
          apply_final_pixel_dc=True,
      )
      channels = image_batch_np.shape[3]
      if batch["hr"].ndim == 3:
          gt = (
              batch["hr"]
              .unsqueeze(1)
              .expand((t2i_config.test_batch_size, channels, 512, 512))
              .to(accelerator.device)
          )
      else:
          gt = (
              batch["hr"]
              .expand((t2i_config.test_batch_size, channels, 512, 512))
              .to(accelerator.device)
          )
      metrics_batch = MRIEvaluator.eval_all_metrics(
          ground_truth=gt,
          generated=torch.from_numpy(image_batch_np)
          .permute(0, 3, 1, 2)
          .to(accelerator.device),
          psnr=psnr,
          ssim=ssim,
      )  # returns hfen, nmse, psnr, ssim
      metrics.append(metrics_batch)
      if idx == 9:
        break

  metrics_np = np.array(metrics)  # shape: (num_batches, 4)
  avg_metrics = metrics_np.mean(axis=0)
  print(f"Metrics are {avg_metrics}")
  # hfen, nmse, psnr, ssim

# Scratch PC Eval

In [ ]:
train_val_test_split = (0.8, 0.1, 0.1)
assert sum(train_val_test_split) == 1.0, "Dataset split should sum up to one"

shared_config: dict = {
    "data_dir": Path("./fastMRI_brain_DICOM"),
    "mode": "train",
    "fractions": train_val_test_split,
    "target_size": (512, 512),
    "contrast_filter": "T2",
    "strength_filter": "3.0T",
    "scale_factor": 4.0,
    "fastMRI_manifest_json": "./fast_MRI_brain_patient_records_manifest_small.json",
}
config = DatasetConfig(**shared_config)
t2i_config = T2IConfig(
    train_batch_size=32,
    test_batch_size=16,
    partial_start_step=700, # sd 1.5 has a noise range of [0,1000]
    max_train_steps=6500,
    pretrained_vae_model_name_or_path="microsoft/mri-autoencoder-v0.1",
)
shared_config["mode"] = "test"
test_dataset = FastMRILazyDataset(config=DatasetConfig(**shared_config))
test_loader = DataLoader(
    test_dataset,
    batch_size=t2i_config.test_batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
accelerator_project_config = ProjectConfiguration(
    project_dir=t2i_config.output_dir, logging_dir=t2i_config.logging_dir
)
accelerator = Accelerator(
    gradient_accumulation_steps=t2i_config.gradient_accumulation_steps,
    mixed_precision=t2i_config.mixed_precision,  # set bf16
    log_with=t2i_config.report_to,
    project_config=accelerator_project_config,
)
set_seed(t2i_config.seed)
os.makedirs(t2i_config.output_dir, exist_ok=True)

weight_dtype = torch.float32
if accelerator.mixed_precision == "fp16":
    weight_dtype = torch.float16
elif accelerator.mixed_precision == "bf16":
    weight_dtype = torch.bfloat16
# load the tokenizers
tokenizer = AutoTokenizer.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="tokenizer",
    revision=t2i_config.revision,
    use_fast=False,
)
# load the correct scheduler and models
text_encoder_cls = import_model_class_from_model_name_or_path(
    t2i_config.pretrained_model_name_or_path,
    t2i_config.revision,
    subfolder="text_encoder",
)
# Load scheduler and models
# timesteps defauls to 1000
noise_scheduler = DDPMScheduler.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="scheduler",
    prediction_type=t2i_config.ddpm_scheduler_prediction_type,  # velocity prediction
    timestep_spacing=t2i_config.ddpm_scheduler_timestep_spacing,  # for zero-SNR
    rescale_betas_zero_snr=t2i_config.ddpm_scheduler_rescale_betas_zero_snr,  # enforces pure noise at t=1000
)
text_encoder = text_encoder_cls.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="text_encoder",
    revision=t2i_config.revision,
    torch_dtype=weight_dtype,
)
vae_path = (
    t2i_config.pretrained_model_name_or_path
    if t2i_config.pretrained_vae_model_name_or_path is None
    else t2i_config.pretrained_vae_model_name_or_path
)
vae = AutoencoderKL.from_pretrained(
    vae_path,
    subfolder="vae" if t2i_config.pretrained_vae_model_name_or_path is None else None,
    revision=t2i_config.revision,
    torch_dtype=weight_dtype,
)

unet_config = UNet2DConditionModel.load_config(
    "segmind/tiny-sd",
    subfolder="unet"
)

new_config = dict(unet_config)
new_config['in_channels'] = 8 # Keep your 8-channel modification
unet = UNet2DConditionModel.from_config(new_config)
unet.train()
# These are never trained to circumvent mode collapse (see controlnet paper)
vae.requires_grad_(False)
text_encoder.requires_grad_(False)

print(
    f"Using VAE: {vae.config['_name_or_path']}, UNET: {unet}, Scheduler Steps: {noise_scheduler.config["num_train_timesteps"]}"
)
if t2i_config.enable_xformers_memory_efficient_attention:
    import xformers # pyright: ignore[reportMissingImports]
    from packaging import version
    xformers_version = version.parse(xformers.__version__)
    if xformers_version == version.parse("0.0.16"):
        print(
            "xFormers 0.0.16 cannot be used for training in some GPUs. If you observe problems during training, please update xFormers to at least 0.0.17. See https://huggingface.co/docs/diffusers/main/en/optimization/xformers for more details."
        )
    unet.enable_xformers_memory_efficient_attention()
    print(f"Enabled efficient attention on UNET")
if t2i_config.gradient_checkpointing:
    unet.enable_gradient_checkpointing()
    print(f"Enabled gradient checkpointing on UNET")
# Enable TF32 for faster training on Ampere GPUs,
# cf https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
if t2i_config.allow_tf32:
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"Enabled matmul.allow_tf32")
if t2i_config.scale_lr:
    learning_rate = (
        t2i_config.learning_rate
        * t2i_config.gradient_accumulation_steps
        * t2i_config.train_batch_size
        * accelerator.num_processes
    )
else:
    learning_rate = t2i_config.learning_rate
# Use 8-bit Adam for lower memory usage or to fine-tune the model in 16GB GPUs
if t2i_config.use_8bit_adam:
    try:
        import bitsandbytes as bnb # pyright: ignore[reportMissingImports]
    except ImportError:
        raise ImportError(
            "To use 8-bit Adam, please install the bitsandbytes library: `pip install bitsandbytes`."
        )
    optimizer_class = bnb.optim.AdamW8bit
    print(f"Enabled 8bit adam")
else:
    optimizer_class = torch.optim.AdamW
is_special_vae = (
    t2i_config.pretrained_vae_model_name_or_path == "microsoft/mri-autoencoder-v0.1"
)
output_dims = 2 if is_special_vae else 3
mri_projector = MRIProjector(output_dims=output_dims).to(accelerator.device)
params_to_optimize = (
    list(unet.parameters())
    + list(mri_projector.parameters())
)
optimizer = optimizer_class(
    params_to_optimize,
    lr=learning_rate,
    betas=(t2i_config.adam_beta1, t2i_config.adam_beta2),
    weight_decay=t2i_config.adam_weight_decay,
    eps=t2i_config.adam_epsilon,
)

# For mixed precision training we cast the text_encoder and vae weights to half-precision
# as these models are only used for inference, keeping weights in full precision is not required.
weight_dtype = torch.float32
if accelerator.mixed_precision == "fp16":
    weight_dtype = torch.float16
elif accelerator.mixed_precision == "bf16":
    weight_dtype = torch.bfloat16

vae.to(accelerator.device, dtype=weight_dtype)
unet.to(accelerator.device, dtype=weight_dtype)
print("Models loaded. VAE in", weight_dtype)
# Let's first compute all the embeddings so that we can free up the text encoders from memory.
# text_encoders = [text_encoder]
tokenizers = [tokenizer]
gc.collect()
torch.cuda.empty_cache()
# Scheduler and math around the number of training steps.
# num_update_steps_per_epoch = math.ceil(len(train_dataloader) / args.gradient_accumulation_steps)
num_update_steps_per_epoch = math.ceil(1e7 / t2i_config.gradient_accumulation_steps)
if t2i_config.max_train_steps is None:
    t2i_config.max_train_steps = (
        t2i_config.num_train_epochs * num_update_steps_per_epoch
    )
lr_scheduler = get_scheduler(
    t2i_config.lr_scheduler_name,
    optimizer=optimizer,
    num_warmup_steps=t2i_config.lr_warmup_steps * accelerator.num_processes,
    num_training_steps=t2i_config.max_train_steps * accelerator.num_processes,
    num_cycles=t2i_config.lr_num_cycles,
    power=t2i_config.lr_power,
)
ema_unet = EMAModel(
    unet.parameters(),
    model_cls=unet.__class__,  # Custom class, passing it here helps with some utilities
    decay=0.9999,
)
# Prepare everything with our `accelerator`.
mri_projector, unet, optimizer, lr_scheduler = accelerator.prepare(
    mri_projector, unet, optimizer, lr_scheduler
)

print_trainable_parameters(unet, name="UNet")
print_trainable_parameters(mri_projector, name="MRI-Projector")

In [ ]:
output_dir = "/content/drive/MyDrive/Colab Notebooks/MasterInfo/GenAI/Scratch + Partial Diffusion + B32 + 4k Steps (helpful-snowball-85)"
mri_projector.load_state_dict(torch.load(os.path.join(output_dir, "mri_projector_scratch.pt")))
unet.load_state_dict(torch.load(os.path.join(output_dir, "unet_scratch.pt")))
gc.collect()
torch.cuda.empty_cache()

In [ ]:
psnr = PeakSignalNoiseRatio(data_range=1.0).to(accelerator.device)
ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(accelerator.device)

from itertools import product

dcs = [1.5, 3.0, 4.5]
tapers = [0.15, 0.3, 0.45]
configs = product(dcs, tapers, [True])
configs = [x for x in configs]
configs.append([1.5, 0.45, False])
for taper, reduction_factor, use_data_consistency in configs:
  print(f"Scratch: evaluating now tau: {taper}, r: {reduction_factor}, use: {use_data_consistency}")
  metrics = []
  for idx, batch in tqdm(enumerate(test_loader), disable=True):
      with torch.no_grad():
          prompt_embeds_eval = compute_embeddings_sd1x5(
              batch=batch,
              proportion_empty_prompts=0.0,  # Use 0.0 for evaluation when skipping CFG
              text_encoders=[text_encoder],
              tokenizers=tokenizers,
              accelerator=accelerator,
          )["prompt_embeds"]
      image_batch_np, postprocessed = generate_mri_slices_partial_latent_align_dc_no_t2i(
        batch=batch,
        mri_projector=mri_projector,
        unet=unet,
        vae=vae,
        noise_scheduler=noise_scheduler,
        prompt_embeds=prompt_embeds_eval,
        start_step=t2i_config.partial_start_step,  # Start from t=250
        num_inference_steps=500,  # Scheduler will slice this
        weight_dtype=weight_dtype,
        accelerator=accelerator,
        use_data_consistency=use_data_consistency,
        dc_reduction_factor=reduction_factor,
        taper=taper,
        apply_final_pixel_dc=True,
      )
      channels = image_batch_np.shape[3]
      if batch["hr"].ndim == 3:
          gt = (
              batch["hr"]
              .unsqueeze(1)
              .expand((t2i_config.test_batch_size, channels, 512, 512))
              .to(accelerator.device)
          )
      else:
          gt = (
              batch["hr"]
              .expand((t2i_config.test_batch_size, channels, 512, 512))
              .to(accelerator.device)
          )
      metrics_batch = MRIEvaluator.eval_all_metrics(
          ground_truth=gt.to(accelerator.device, dtype=torch.float32),
          generated=torch.from_numpy(image_batch_np)
          .permute(0, 3, 1, 2)
          .to(accelerator.device, dtype=torch.float32),
          psnr=psnr,
          ssim=ssim,
      )  # returns hfen, nmse, psnr, ssim
      metrics.append(metrics_batch)
      if idx == 9:
        break
      
metrics_np = np.array(metrics)  # shape: (num_batches, 4)
avg_metrics = metrics_np.mean(axis=0)
print(f"Metrics are {avg_metrics}")

# ControlNet PC Eval

In [ ]:
train_val_test_split = (0.8, 0.1, 0.1)
assert sum(train_val_test_split) == 1.0, "Dataset split should sum up to one"

shared_config: dict = {
    "data_dir": Path("./fastMRI_brain_DICOM"),
    "mode": "train",
    "fractions": train_val_test_split,
    "target_size": (512, 512),
    "contrast_filter": "T2",
    "strength_filter": "3.0T",
    "scale_factor": 4.0,
    "fastMRI_manifest_json": "./fast_MRI_brain_patient_records_manifest_small.json",
}
config = DatasetConfig(**shared_config)
t2i_config = T2IConfig(
    train_batch_size=32,
    test_batch_size=16,
    partial_start_step=700, # sd 1.5 has a noise range of [0,1000]
    max_train_steps=6500,
    pretrained_vae_model_name_or_path="microsoft/mri-autoencoder-v0.1",
)
shared_config["mode"] = "test"
test_dataset = FastMRILazyDataset(config=DatasetConfig(**shared_config))
test_loader = DataLoader(
    test_dataset,
    batch_size=t2i_config.test_batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
accelerator_project_config = ProjectConfiguration(
    project_dir=t2i_config.output_dir, logging_dir=t2i_config.logging_dir
)
accelerator = Accelerator(
    gradient_accumulation_steps=t2i_config.gradient_accumulation_steps,
    mixed_precision=t2i_config.mixed_precision,
    log_with=t2i_config.report_to,
    project_config=accelerator_project_config,
)
set_seed(t2i_config.seed)
os.makedirs(t2i_config.output_dir, exist_ok=True)

weight_dtype = torch.float32
if accelerator.mixed_precision == "fp16":
    weight_dtype = torch.float16
elif accelerator.mixed_precision == "bf16":
    weight_dtype = torch.bfloat16
tokenizer = AutoTokenizer.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="tokenizer",
    revision=t2i_config.revision,
    use_fast=False,
)
text_encoder_cls = import_model_class_from_model_name_or_path(
    t2i_config.pretrained_model_name_or_path,
    t2i_config.revision,
    subfolder="text_encoder",
)
noise_scheduler = DDPMScheduler.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="scheduler",
    prediction_type=t2i_config.ddpm_scheduler_prediction_type,
    timestep_spacing=t2i_config.ddpm_scheduler_timestep_spacing,
    rescale_betas_zero_snr=t2i_config.ddpm_scheduler_rescale_betas_zero_snr,
)
text_encoder = text_encoder_cls.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="text_encoder",
    revision=t2i_config.revision,
    torch_dtype=weight_dtype,
)
vae_encoder = AutoencoderKL.from_pretrained(
    t2i_config.pretrained_vae_model_name_or_path,
    subfolder=None,
    revision=t2i_config.revision,
    torch_dtype=weight_dtype,
)
vae_decoder = AutoencoderKL.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="vae",
    revision=t2i_config.revision,
    torch_dtype=weight_dtype,
)
unet = UNet2DConditionModel.from_pretrained(
    t2i_config.pretrained_model_name_or_path,
    subfolder="unet",
    revision=t2i_config.revision,
    torch_dtype=weight_dtype,
)
vae_encoder.requires_grad_(False)
vae_decoder.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

print(
    f"Using VAE-Encoder: {vae_encoder.config['_name_or_path']}, VAE-Decoder: {vae_decoder.config['_name_or_path']}, UNET: {unet.config['_name_or_path']}, Scheduler Steps: {noise_scheduler.config['num_train_timesteps']}"
)
if t2i_config.enable_xformers_memory_efficient_attention:
    import xformers  # pyright: ignore[reportMissingImports]
    from packaging import version
    xformers_version = version.parse(xformers.__version__)
    if xformers_version == version.parse("0.0.16"):
        print(
            "xFormers 0.0.16 cannot be used for training in some GPUs. If you observe problems during training, please update xFormers to at least 0.0.17. See https://huggingface.co/docs/diffusers/main/en/optimization/xformers for more details."
        )
    unet.enable_xformers_memory_efficient_attention()
    print(f"Enabled efficient attention on UNET")
if t2i_config.gradient_checkpointing:
    unet.enable_gradient_checkpointing()
    print(f"Enabled gradient checkpointing on UNET")
if t2i_config.allow_tf32:
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"Enabled matmul.allow_tf32")
if t2i_config.scale_lr:
    learning_rate = (
        t2i_config.learning_rate
        * t2i_config.gradient_accumulation_steps
        * t2i_config.train_batch_size
        * accelerator.num_processes
    )
else:
    learning_rate = t2i_config.learning_rate
if t2i_config.use_8bit_adam:
    try:
        import bitsandbytes as bnb  # pyright: ignore[reportMissingImports]
    except ImportError:
        raise ImportError(
            "To use 8-bit Adam, please install the bitsandbytes library: `pip install bitsandbytes`."
        )
    optimizer_class = bnb.optim.AdamW8bit
    print(f"Enabled 8bit adam")
else:
    optimizer_class = torch.optim.AdamW
is_special_vae = (
    t2i_config.pretrained_vae_model_name_or_path == "microsoft/mri-autoencoder-v0.1"
)
output_dims = 2 if is_special_vae else 3

# ControlNet initialized from the frozen UNet weights
controlnet = ControlNetModel.from_unet(unet, conditioning_channels=2)
controlnet.enable_gradient_checkpointing()
controlnet = controlnet.to(accelerator.device)

latent_projector = LatentMRIProjector(
    spatial_in=128, spatial_out=64, in_channels=4, out_channels=4
).to(accelerator.device)
mri_projector = MRIProjector(output_dims=output_dims).to(accelerator.device)

params_to_optimize = (
    list(controlnet.parameters())
    + list(latent_projector.parameters())
    + list(mri_projector.parameters())
)
optimizer = optimizer_class(
    params_to_optimize,
    lr=learning_rate,
    betas=(t2i_config.adam_beta1, t2i_config.adam_beta2),
    weight_decay=t2i_config.adam_weight_decay,
    eps=t2i_config.adam_epsilon,
)

weight_dtype = torch.float32
if accelerator.mixed_precision == "fp16":
    weight_dtype = torch.float16
elif accelerator.mixed_precision == "bf16":
    weight_dtype = torch.bfloat16

vae_encoder.to(accelerator.device, dtype=weight_dtype)
vae_decoder.to(accelerator.device, dtype=weight_dtype)
unet.to(accelerator.device, dtype=weight_dtype)
print("Models loaded. UNet and VAE in", weight_dtype)

print_trainable_parameters(controlnet, name="ControlNet")
print_trainable_parameters(latent_projector, name="MRI Latent Projector")
print_trainable_parameters(mri_projector, name="MRI-Projector")

tokenizers = [tokenizer]
gc.collect()
torch.cuda.empty_cache()
num_update_steps_per_epoch = math.ceil(1e7 / t2i_config.gradient_accumulation_steps)
if t2i_config.max_train_steps is None:
    t2i_config.max_train_steps = (
        t2i_config.num_train_epochs * num_update_steps_per_epoch
    )
lr_scheduler = get_scheduler(
    t2i_config.lr_scheduler_name,
    optimizer=optimizer,
    num_warmup_steps=t2i_config.lr_warmup_steps * accelerator.num_processes,
    num_training_steps=t2i_config.max_train_steps * accelerator.num_processes,
    num_cycles=t2i_config.lr_num_cycles,
    power=t2i_config.lr_power,
)
ema_controlnet = EMAModel(
    controlnet.parameters(),
    model_cls=controlnet.__class__,
    decay=0.9999,
)
controlnet, latent_projector, mri_projector, unet, optimizer, lr_scheduler = accelerator.prepare(
    controlnet, latent_projector, mri_projector, unet, optimizer, lr_scheduler
)

print_trainable_parameters(latent_projector, name="MRI Latent Projector")
print_trainable_parameters(controlnet, name="ControlNet")
print_trainable_parameters(mri_projector, name="MRI-Projector")

In [ ]:
output_dir = "/content/drive/MyDrive/Colab Notebooks/MasterInfo/GenAI/controlnet_partial_65k"
mri_projector.load_state_dict(torch.load(os.path.join(output_dir, "mri_projector.pt")))
latent_projector.load_state_dict(torch.load(os.path.join(output_dir, "latent_projector.pt")))
controlnet.load_state_dict(torch.load(os.path.join(output_dir, "controllnet_tuned.pt")))
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from itertools import product
from src.t2iadapter.utils import generate_mri_slices_partial_latent_align_dc_controlnet

psnr = PeakSignalNoiseRatio(data_range=1.0).to(accelerator.device)
ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(accelerator.device)

dcs = [1.5, 3.0, 4.5]
tapers = [0.15, 0.3, 0.45]
configs = product(dcs, tapers, [True])
configs = [x for x in configs]
configs.append([1.5, 0.45, False])
for taper, reduction_factor, use_data_consistency in configs:
  print(f"LoRA: evaluating now tau: {taper}, r: {reduction_factor}, use: {use_data_consistency}")
  metrics = []
  for idx, batch in tqdm(enumerate(test_loader), disable=True):
      with torch.no_grad():
          prompt_embeds_eval = compute_embeddings_sd1x5(
              batch=batch,
              proportion_empty_prompts=0.0,  # Use 0.0 for evaluation when skipping CFG
              text_encoders=[text_encoder],
              tokenizers=tokenizers,
              accelerator=accelerator,
          )["prompt_embeds"]
      images_batch_np, postprocessed = generate_mri_slices_partial_latent_align_dc_controlnet(
          batch=batch,
          controlnet=controlnet,
          mri_projector=mri_projector,
          latent_projector=latent_projector,
          unet=unet,
          vae_encoder=vae_encoder,
          vae_decoder=vae_decoder,
          noise_scheduler=noise_scheduler,
          prompt_embeds=prompt_embeds_eval,
          start_step=t2i_config.partial_start_step,
          num_inference_steps=500,
          weight_dtype=weight_dtype,
          accelerator=accelerator,
          use_data_consistency=use_data_consistency,
          dc_reduction_factor=reduction_factor,
          taper=taper,
          apply_final_pixel_dc=True,
      )
      images_batch_np = 1 - images_batch_np
      channels = images_batch_np.shape[3]
      if batch["hr"].ndim == 3:
          gt = (
              batch["hr"]
              .unsqueeze(1)
              .expand((t2i_config.test_batch_size, channels, 512, 512))
              .to(accelerator.device)
          )
      else:
          gt = (
              batch["hr"]
              .expand((t2i_config.test_batch_size, channels, 512, 512))
              .to(accelerator.device)
          )
      metrics_batch = MRIEvaluator.eval_all_metrics(
          ground_truth=gt,
          generated=torch.from_numpy(images_batch_np)
          .permute(0, 3, 1, 2)
          .to(accelerator.device),
          psnr=psnr,
          ssim=ssim,
      )  # returns hfen, nmse, psnr, ssim
      metrics.append(metrics_batch)
      if idx == 9:
        break
metrics_np = np.array(metrics)  # shape: (num_batches, 4)
avg_metrics = metrics_np.mean(axis=0)
print(f"Metrics are {avg_metrics}")
# hfen, nmse, psnr, ssim